# Red Cedar — Training in Depth

A closer look at training, beyond the quick start:
1. Layers and layer chaining
2. Test accuracy over the course of training
3. Results per class and per layer
4. Choosing the train and test data
5. Finding and reusing trained models
6. Checking on and stopping a running job

Run the setup cell first; after that, each section can be run on its own.

In [ ]:
import redcedar
import matplotlib.pyplot as plt

# Connect to the server
# Reads REDCEDAR_URL and REDCEDAR_API_KEY from environment, or pass explicitly:
# client = redcedar.connect(url="http://localhost:8002", api_key="your-key")
client = redcedar.connect()
print(f"Red Cedar SDK v{redcedar.__version__}")
print(f"Server: {client.url}")
print(f"Status: {client.status()}")

dataset = "wdbc"
base = {"num_qubits": 6, "seed": 123}

## 1. Layers and layer chaining

A training run covers the layers from `min_layers` to `max_layers`. Each layer is a separate model with one more layer of operators than the one before.

With `chain_layers` on (the default), each layer starts from the parameters the previous layer found, with its new operators starting at zero, so it begins exactly where the smaller model left off. With `chain_layers` off, each layer starts from scratch. Below, the same three layers are trained both ways (this takes a few minutes).

In [ ]:
runs = {}
for chain in [True, False]:
    runs[chain] = client.training.run(dataset, settings=dict(base, min_layers=1, max_layers=3, chain_layers=chain))
    print(f"chain_layers={chain}:")
    for layer, v in runs[chain]["validation"].items():
        print(f"  layer {layer}: test accuracy {v['test_accuracy']:.1%}, test loss {v['test_loss']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, chain in zip(axes, [True, False]):
    start = 0
    for layer, points in runs[chain]["history"].items():
        ax.plot(range(start, start + len(points)), [p["train_loss"] for p in points], label=f"layer {layer}")
        start += len(points)
    ax.set(title=f"chain_layers={chain}", xlabel="Parameter update", ylabel="Training loss")
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Test accuracy over the course of training

The server saves the parameters after every update. `training.analyze()` replays them on the test data, so you can compare test accuracy with training accuracy as training goes on, and see whether the model starts to overfit. `layer` picks the layer (default: the last); here it's layer 1, where most of the learning happens.

In [ ]:
train_id = runs[True]["train_id"]
replay = client.training.analyze(train_id, layer=1)
train_curve = runs[True]["history"][1]

print(f"Layer {replay['layer']}: {replay['num_checkpoints']} checkpoints, "
      f"final test accuracy {replay['final_accuracy']:.1%}")

plt.figure(figsize=(7, 4))
plt.plot([p["train_accuracy"] for p in train_curve], label="training data")
plt.plot(replay["accuracy_history"], label="test data")
plt.xlabel("Parameter update")
plt.ylabel("Accuracy")
plt.title("Layer 1")
plt.legend()
plt.grid(True)
plt.show()

## 3. Results per class and per layer

The validation results for each layer include the test accuracy for each class. `training.evaluate()` can also test any single layer of a trained model.

In [ ]:
for layer, v in runs[True]["validation"].items():
    per_class = ", ".join(f"class {c}: {a:.1%}" for c, a in v["test_accuracy_per_class"].items())
    print(f"Layer {layer}: {per_class}")

layer_1 = client.training.evaluate(train_id, layer=1)
print(f"\nInference on layer 1: accuracy {layer_1['overall_accuracy']:.1%} on {layer_1['test_samples']} samples")

## 4. Choosing the train and test data

`train_size` and `test_size` set how much data is used, as a fraction of the dataset or a number of samples. `class_range` picks a subset of classes in a multi-class dataset (catalog datasets only). Smaller training sets train faster; larger test sets give more reliable accuracy figures.

In [ ]:
small = client.training.run(dataset, settings=dict(base, max_layers=1), train_size=100, test_size=200)
check = client.training.evaluate(small["train_id"])
print(f"Trained on 100 samples: test accuracy {check['overall_accuracy']:.1%} on {check['test_samples']} test samples")

## 5. Finding and reusing trained models

Trained models stay on the server, so a `train_id` can be used later, from another notebook or session, for inference, explainable AI or the replay. `inference.models()` lists the most recent models with the settings they were trained with.

In [ ]:
for m in client.inference.models()["models"]:
    p = m["training_params"]
    print(f"{m['train_id'][:8]}  {m['dataset']:18s} qubits={m['num_qubits']}  layers={m['layers']}  seed={p.get('seed')}")

## 6. Checking on and stopping a running job

`training.run()` waits until training ends. To keep working while a model trains, start it in a background thread; `training.status()` reports on it and `training.stop()` stops it. Layers that finished are kept. The server runs one job at a time, so a job submitted while another runs waits in the queue and starts when the first one ends.

In [ ]:
import threading, time

job = {}
def train_in_background():
    try:
        job["result"] = client.training.run(dataset, settings={"num_qubits": 6, "min_layers": 1, "max_layers": 3})
    except RuntimeError as e:
        job["error"] = str(e)

thread = threading.Thread(target=train_in_background)
thread.start()

# Wait until the job is running (the SDK keeps the ID of the job it submitted last)
time.sleep(2)
while True:
    try:
        status = client.training.status(client._last_job_id)
        if status["status"] == "running":
            break
    except Exception:
        pass   # not started yet
    time.sleep(1)
print("Status:", status["status"], f"({status['progress']}%)")

time.sleep(5)
print(client.training.stop(client._last_job_id)["message"])
thread.join()
print("training.run() ended with:", job)
print("Status:", client.training.status(client._last_job_id)["status"])